# Analysis and Plotting

In [ ]:
#libraries
import pandas as pd
import numpy as np
import os
import sys
import joblib
import warnings
import pickle
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import LeavePGroupsOut
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, norm
import scipy.stats as st
from scipy.stats import zscore as  zscore
from joblib import Parallel, delayed
# from multiprocessing import Manager
from sklearn.cross_decomposition import PLSRegression
import gc
import traceback
import logging
from sklearn.model_selection import KFold
import random
from scipy.stats import gaussian_kde
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors


In [ ]:
# Unimodal lables
l1_labels = {
    'conn_mid': 'MID FC',
    'gfc': 'General FC',
    'tfc': 'Multitask FC',
    'conn_rest': 'Rest FC', 
    'conn_wm': 'Nback FC',
    'conn_sst': 'SST FC',
    'T2_gray': 'T2 Gray Matter Avg Intensity',
    'subnet_rest': 'Rest Subcortical-Net FC',
    'cntr_Stop-CorrectGo_task-SST': 'SST Stop-CorrectGo',
    'T1_gray': 'T1 Gray Matter Avg Intensity',
    'T2_white': 'T2 White Matter Avg Intensity',
    'cntr_IncorrectStop-CorrectGo_task-SST': 'SST IncorrectStop-CorrectGo',
    'cntr_IncorrectGo_task-SST': 'SST IncorrectGo',
    'cntr_IncorrectStop_task-SST': 'SST IncorrectStop',
    'cntr_IncorrectGo-CorrectGo_task-SST': 'SST IncorrectGo-CorrectGo',
    'T1_norm': 'T1 Normalised Intensity',
    'cntr_CorrectStop_task-SST': 'SST CorrectStop',
    'T1_white': 'T1 White Matter Avg Intensity',
    'T2_norm': 'T2 Normalised Intensity',
    'cntr_CorrectStop-CorrectGo_task-SST': 'SST CorrectStop-CorrectGo',
    'surf': 'Surface Area',
    'cntr_IncorrectGo-IncorrectStop_task-SST': 'SST IncorrectGo-IncorrectStop',
    'cntr_CorrectGo_task-SST': 'SST CorrectGo',
    'cntr_CorrectStop-IncorrectStop_task-SST': 'SST CorrectStop-IncorrectStop',
    'T1_summ': 'T1 Summations',
    'VolBrain': 'FreeSurfer Summations',
    'DTI': 'DTI',
    'T2_summ': 'T2 Summations',
    'Sulcal_Depth': 'Sulcal Depth',
    'Avg_T1_ASEG_Vol_': 'T1 Subcortical Volume',
    'Dest_Thick_': 'Cortical Thickness',
    'Avg_T2_ASEG_': 'T2 Subcortical Volume',
    'rsmri_within_avg_data': 'Rest cortical-Net FC',
    'Dest_Vol_': 'Cortical Volume',
    'rsmri_gordon_aseg_data': 'Rest Temporal Variance',
    'antiLargeRewVsSmallRew_ROI_mid': 'MID LargeReward-SmallReward',
    'feedPunPosVsNeg_ROI_mid': 'MID LossHit-LossMiss',
    'incorrectgovsincorrectstop_ROI_sst': 'SST IncorrectGo-IncorrectStop',
    'anystopvscorrectgo_ROI_sst': 'SST AnyStop-CorrectStop',
    'incorrectstopvscorrectgo_ROI_sst': 'SST IncorrectStop-CorrectGo',
    'emotionvsneutface_ROI_nbk': 'Nback EmotionFace-NeutFace',
    'incorrectgovscorrectgo_ROI_sst': 'SST IncorrectGo-CorrectGo',
    'correctgovsfixation_ROI_sst': 'SST CorrectGo-Fixation',
    'antiRewVsNeu_ROI_mid': 'MID Reward-Neutral',
    'X2back_ROI_nbk': 'Nback 2back',
    'facevsplace_ROI_nbk': 'Nback Face-Place',
    'antiSmallLossVsNeu_ROI_mid': 'MID SmallLoss-Neutral',
    'negfacevsneutface_ROI_nbk': 'Nback NegFace-NeutFace',
    'antiLargeLossVsNeu_ROI_mid': 'MID LargeLoss-Neutral',
    'emotion_ROI_nbk': 'Nback EmotionFace',
    'correctstopvsincorrectstop_ROI_sst': 'SST CorrectStop-IncorrectStop',
    'antiSmallRewVsNeu_ROI_mid': 'MID SmallReward-Neutral',
    'antiLargeRewVsNeu_ROI_mid': 'MID LargeReward-Neutral',
    'antiLosVsNeu_ROI_mid': 'MID Loss-Neutral',
    'correctstopvscorrectgo_ROI_sst': 'SST CorrectStop-CorrectGo',
    'posfacevsneutface_ROI_nbk': 'Nback PosFace-NeutFace',
    'X0back_ROI_nbk': 'Nback 0back',
    'place_ROI_nbk': 'Nback Place',
    'antiLargeLossVsSmallLoss_ROI_mid': 'MID LargeLoss-SmallLoss',
    'X2backvs0back_ROI_nbk': 'Nback 2-0back',
    'feedRewPosVsNeg_ROI_mid': 'MID RewardHit-RewardMiss',
    'artr_twobk_task-nback': 'Nback 2back',
    'artr_LossHit-LossMiss_task-MID': 'MID LossHit-LossMiss',
    'artr_SmallLoss-Neutral_task-MID': 'MID SmallLoss-Neutral',
    'artr_LgReward-SmallReward_task-MID': 'MID LargeReward-SmallReward',
    'artr_Loss-Neutral_task-MID': 'MID Loss-Neutral',
    'artr_PosFace-NeutFace_task-nback': 'Nback PosFace-NeutFace',
    'artr_LgLoss-Neutral_task-MID': 'MID LargeLoss-Neutral',
    'artr_NegFace-NeutFace_task-nback': 'Nback NegFace-NeutFace',
    'artr_face-place_task-nback': 'Nback Face-Place',
    'artr_LgReward-Neutral_task-MID': 'MID LargeReward-Neutral',
    'artr_RewardHit-RewardMiss_task-MID': 'MID RewardHit-RewardMiss',
    'artr_place_task-nback': 'Nback Place',
    'artr_twobk-zerobk_task-nback': 'Nback 2-0back',
    'artr_LgLoss-SmallLoss_task-MID': 'MID LargeLoss-SmallLoss',
    'artr_face_task-nback': 'Nback Face',
    'artr_emotionface_task-nback': 'Nback EmotionFace',
    'artr_emotionface-NeutFace_task-nback': 'Nback Emotionface-NeutFace',
    'artr_zerobk_task-nback': 'Nback 0back',
    'artr_SmallReward-Neutral_task-MID': 'MID SmallReward-Neutral',
    'artr_Reward-Neutral_task-MID': 'MID Reward-Neutral'
}


In [ ]:
# multimodal labels
#stack model label 
stacked_labels = {
    'nAll1_abcd' : 'Stacked all', 
    'AllRest' : 'Rest FCs', #
    'nGD_TaskCntr' : 'Task Contrasts',#'cntr', just Glasser
    'AllSmri' : 'sMRI',#'abcc+abcd smri',
    'NonTask' : 'Non-Task',#'abcc+abcd smri + rest FC',
    'AllFCs' : 'All FCs',
    'TaskConn' : 'Task FCs',
    'nGD_TaskAll' : 'Task FCs + Contrasts',
    
    'nGD_WMCntr' : 'Nback Contrasts',#Glass
    'GD_SstCntr' : 'SST Contrasts',#Glass
    'nGD_MidCntr' : 'MID Contrasts', #Glass
    'nGD_WMAll' : 'Nback Contrasts + FC', # glasser
    'nGD_SstAll' : 'SST Contrasts + FC', # Glasser
    'nGD_MidAll' : 'MID Contrasts + FC', #Glasser
}

## Calculate performance metrics 

### Unimodal

In [ ]:
### concatenate predicted values across test folds + calculate performance metrics 
import os
import joblib
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

# Paths and settings
abcd_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/'
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
n_folds = 2
P = 1000  # Number of permutation iterations
n_jobs = -1

# Target variables and modality chunks
targets = ['nihtbx_totalcomp_uncorrected', 'nihtbx_cryst_uncorrected', 'nihtbx_fluidcomp_uncorrected']
target_names = ['total_']#, 'cryst_', 'fluid_'
modality_chunks = ['abccConn', 'abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']
# training strategy:
model_types = {'All': '_pls_output_std_All', 'wa': '_pls_output_std_wa', 
               'aa': '_pls_output_std_aa', 'aawa': '_pls_output_std_waaa'}
# model_types = {
#                 'All': '_pls_output_std_All', 'wa': '_pls_output_std_wa', 
#                 'aa': '_pls_output_std_aa', 'aawa': '_pls_output_std_waaa',
#                 'ha': '_pls_output_std_ha', 'hawa': '_pls_output_std_ha'}
# model_types = {
#                 'wa': '_pls_output_std_wa', 
#                 'aa': '_pls_output_std_aa',
#                 'ha': '_pls_output_std_ha'}
# model_types = {'All': '_krr_output_std_All', 'wa': '_krr_output_std_wa', 
#                'aa': '_krr_output_std_aa', 'aawa': '_krr_output_std_waaa'}
# Load demographic data
demo = pd.read_csv(abcd_dir + 'demo_nesi.csv', index_col=0).dropna()

# Statistical metric functions
def r2_statistic(y_true, y_pred, y_train=None):
    return r2_score(y_true, y_pred)

def cod_statistic(y_true, y_pred, y_train=None):
    # If y_train is provided, compute SST from training data
    if y_train is not None:
        sst = np.sum((y_train - np.mean(y_train))**2) / len(y_train)
    else:
        sst = np.sum((y_true - np.mean(y_true))**2) / len(y_true)
    sse = np.sum((y_true - y_pred)**2) / len(y_true)
    cod = 1 - sse / sst if sst != 0 else 0
    return cod

def pearson_statistic(y_true, y_pred, y_train=None):
    r, _ = pearsonr(y_true, y_pred)
    # Apply Fisher's z-transformation
    z = np.arctanh(r)
    return z
def mse_statistic(y_true, y_pred, y_train=None):
    return mean_squared_error(y_true, y_pred)

def mae_statistic(y_true, y_pred, y_train=None):
    return mean_absolute_error(y_true, y_pred)
# Metric registry
METRIC_FUNCTIONS = {
    'r2': r2_statistic,
    'cod': cod_statistic,
    'pearson': pearson_statistic,
    'mse': mse_statistic,
    'mae': mae_statistic
}

# compute metric difference
def metric_difference_statistic(y_true, y_pred, aa_indices, wa_indices, index_map, metric_func, y_train=None):
    aa_pos = [index_map[idx] for idx in aa_indices]
    wa_pos = [index_map[idx] for idx in wa_indices]

    y_true_aa = y_true[aa_pos]
    y_pred_aa = y_pred[aa_pos]
    y_true_wa = y_true[wa_pos]
    y_pred_wa = y_pred[wa_pos]

    metric_aa = metric_func(y_true_aa, y_pred_aa, y_train)
    metric_wa = metric_func(y_true_wa, y_pred_wa, y_train)

    return metric_aa - metric_wa

# permutation test
def permutation_metric_difference(y_true, y_pred, aa_indices, wa_indices, metric_func, y_train=None, n_permutations=100):
    index_map = {idx: i for i, idx in enumerate(aa_indices + wa_indices)}
    observed_diff = metric_difference_statistic(y_true, y_pred, aa_indices, wa_indices, index_map, metric_func, y_train)
    perm_diffs = []
    rng = np.random.default_rng(42)
    for _ in range(n_permutations):
        perm_labels = rng.permutation(aa_indices + wa_indices)
        perm_aa = perm_labels[:len(aa_indices)]
        perm_wa = perm_labels[len(aa_indices):]
        perm_diff = metric_difference_statistic(y_true, y_pred, perm_aa, perm_wa, index_map, metric_func, y_train)
        perm_diffs.append(perm_diff)
    perm_diffs = np.array(perm_diffs)
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
    return perm_diffs, p_value

# Main analysis:
def run_statistical_analysis(metric_type):
    if metric_type not in METRIC_FUNCTIONS:
        raise ValueError(f"Unsupported metric type: {metric_type}. Choose from {list(METRIC_FUNCTIONS.keys())}")
    metric_func = METRIC_FUNCTIONS[metric_type]
    
    results = {metric_type: {}}
    for targ, target_name in zip(targets, target_names):
        results[metric_type][targ] = {}
        for features_name in modality_chunks:
            results[metric_type][targ][features_name] = {}
            # Load feature data
            featuresd = joblib.load(f'{fold_base_path}Fold_0/pls/{target_name}{features_name}_pls_output_std_All.joblib')
            feature_keys = list(featuresd.keys())
            for key in feature_keys:
                results[metric_type][targ][features_name][key] = {}
                for model_name, suffix in model_types.items():
                    # Concatenate data across folds
                    y_true_all = []
                    y_pred_all = []
                    y_train_all = []
                    for fold_num in range(n_folds):
                        pls_dir = f'{fold_base_path}Fold_{fold_num}/pls/'
                        file_path = f'{pls_dir}{target_name}{features_name}{suffix}.joblib'
                        if not os.path.exists(file_path):
                            print(f"Missing file: {file_path}")
                            continue
                        pls_dict = joblib.load(file_path)
                        if key not in pls_dict:
                            print(f"Feature set {key} not found in {file_path}")
                            continue
                        y_true_all.append(pls_dict[key]['data'][test_t])
                        y_pred_all.append(pls_dict[key]['data'][test_p])
                        if 'yttrain' in pls_dict[key]['data']:
                            y_train_all.append(pls_dict[key]['data']['yttrain'])
                    
                    if not y_true_all or not y_pred_all:
                        print(f"No valid data for {targ}, {features_name}, {key}, {model_name}. Skipping.")
                        continue
                    
                    # Concatenate data
                    y_true_all = pd.concat(y_true_all)
                    y_pred_all = pd.concat(y_pred_all)
                    y_train_all = pd.concat(y_train_all) if y_train_all else None
                    
                    # Get indices
                    indices_all = y_true_all.index
                    common_ind = y_true_all.index.intersection(demo.index)
                    demo_subset = demo.loc[common_ind]
                    aa_indices = demo_subset[demo_subset['race_ethnicity'] == 2].index.tolist()
                    wa_indices = demo_subset[demo_subset['race_ethnicity'] == 1].index.tolist()
                    ha_indices = demo_subset[demo_subset['race_ethnicity'] == 3].index.tolist()
                    all_indices = aa_indices + wa_indices + ha_indices
                    
                    # Create index mapping for NumPy arrays
                    y_true_all = y_true_all.loc[all_indices]
                    y_pred_all = y_pred_all.loc[all_indices]
                    index_map = {idx: i for i, idx in enumerate(all_indices)}
                    aa_pos = [index_map[idx] for idx in aa_indices]
                    wa_pos = [index_map[idx] for idx in wa_indices]
                    
                    # Convert to NumPy arrays
                    y_true_array = y_true_all.values.flatten()
                    y_pred_array = y_pred_all.values.flatten()
                    y_train_array = y_train_all.values.flatten() if y_train_all is not None else None
                    
                    # Compute metrics on concatenated data
                    metric_aa = metric_func(y_true_array[aa_pos], y_pred_array[aa_pos], y_train_array)
                    metric_wa = metric_func(y_true_array[wa_pos], y_pred_array[wa_pos], y_train_array)
                    diff_aa = metric_aa - metric_wa
                    
                    # Permutation test
                    perm_diffs_aa, perm_p_value_aa = permutation_metric_difference(
                        y_true_array, y_pred_array, aa_indices, wa_indices, metric_func, y_train_array, n_permutations=P
                    )

                    # Compute mean and std for perm_diffs
                    perm_diff_mean_aa = np.mean(perm_diffs_aa)
                    perm_diff_std_aa = np.std(perm_diffs_aa, ddof=1)

                    # save to dict
                    results[metric_type][targ][features_name][key][model_name] = {
                        'metric_aa': metric_aa,
                        'metric_wa': metric_wa,
                        'diff_aa': diff_aa,
                        'perm_diff_mean_aa': perm_diff_mean_aa,
                        'perm_diff_std_aa': perm_diff_std_aa,
                        'perm_diff_dist_aa': perm_diffs_aa,
                        'perm_p_value_aa': perm_p_value_aa,
                    }
    
    # Save results with metric type in filename
    output_dir = os.path.join(fold_base_path, 'results_concat_transformed')
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(results, os.path.join(output_dir, f'pls_stat_results_{metric_type}.joblib'))
    print(f"Statistical comparison for {metric_type} completed and saved.")
    return results


if __name__ == "__main__":
    test_p = 'yptest'
    test_t = 'yttest'
    metrics = ['mae']#'r2', 'cod', 'pearson', 'mse', 

    Parallel(n_jobs=5)(
    delayed(run_statistical_analysis)(metric_type=metric)
    for metric in metrics)
    

### Multimodal

In [ ]:
### concatenate predicted values across test folds + calculate performance metrics 

# Paths and settings
abcd_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/'
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
n_folds = 2
P = 1000  # Number of permutation iterations
n_jobs = -1

# Target variables and modality chunks
targets = ['nihtbx_totalcomp_uncorrected', 'nihtbx_cryst_uncorrected', 'nihtbx_fluidcomp_uncorrected']
target_names = ['total_', 'cryst_', 'fluid_']
modality_chunks = ['abccConn','abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']
# model_types = {'All': '_pls_output_std_All', 'wa': '_pls_output_std_wa', 
#                'aa': '_pls_output_std_aa', 'aawa': '_pls_output_std_waaa'}
model_types = {'All': 'rf2-1_output_xstd_All', 'wa': 'rf2-1_output_xstd_wa', 
               'aa': 'rf2-1_output_xstd_aa', 'aawa': 'rf2-1_output_xstd_waaa'}
# Load demographic data
demo = pd.read_csv(abcd_dir + 'demo_nesi.csv', index_col=0).dropna()

# Statistical metric functions
def r2_statistic(y_true, y_pred, y_train=None):
    return r2_score(y_true, y_pred)

def cod_statistic(y_true, y_pred, y_train=None):
    # If y_train is provided, compute SST from training data
    if y_train is not None:
        sst = np.sum((y_train - np.mean(y_train))**2) / len(y_train)
    else:
        sst = np.sum((y_true - np.mean(y_true))**2) / len(y_true)
    sse = np.sum((y_true - y_pred)**2) / len(y_true)
    cod = 1 - sse / sst if sst != 0 else 0
    return cod

def pearson_statistic(y_true, y_pred, y_train=None):
    r, _ = pearsonr(y_true, y_pred)
    # Apply Fisher's z-transformation
    z = np.arctanh(r)
    return z
def mse_statistic(y_true, y_pred, y_train=None):
    return mean_squared_error(y_true, y_pred)

def mae_statistic(y_true, y_pred, y_train=None):
    return mean_absolute_error(y_true, y_pred)
# Metric registry
METRIC_FUNCTIONS = {
    'r2': r2_statistic,
    'cod': cod_statistic,
    'pearson': pearson_statistic,
    'mse': mse_statistic,
    'mae': mae_statistic
}

# compute metric difference
def metric_difference_statistic(y_true, y_pred, aa_indices, wa_indices, index_map, metric_func, y_train=None):
    aa_pos = [index_map[idx] for idx in aa_indices]
    wa_pos = [index_map[idx] for idx in wa_indices]
    y_true_aa = y_true[aa_pos]
    y_pred_aa = y_pred[aa_pos]
    y_true_wa = y_true[wa_pos]
    y_pred_wa = y_pred[wa_pos]
    metric_aa = metric_func(y_true_aa, y_pred_aa, y_train)
    metric_wa = metric_func(y_true_wa, y_pred_wa, y_train)
    return metric_aa - metric_wa

# permutation test
def permutation_metric_difference(y_true, y_pred, aa_indices, wa_indices, metric_func, y_train=None, n_permutations=100):
    index_map = {idx: i for i, idx in enumerate(aa_indices + wa_indices)}
    observed_diff = metric_difference_statistic(y_true, y_pred, aa_indices, wa_indices, index_map, metric_func, y_train)
    perm_diffs = []
    rng = np.random.default_rng(42)
    for _ in range(n_permutations):
        perm_labels = rng.permutation(aa_indices + wa_indices)
        perm_aa = perm_labels[:len(aa_indices)]
        perm_wa = perm_labels[len(aa_indices):]
        perm_diff = metric_difference_statistic(y_true, y_pred, perm_aa, perm_wa, index_map, metric_func, y_train)
        perm_diffs.append(perm_diff)
    perm_diffs = np.array(perm_diffs)
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
    return perm_diffs, p_value

# Main analysis:
def run_statistical_analysis(metric_type='r2'):
    if metric_type not in METRIC_FUNCTIONS:
        raise ValueError(f"Unsupported metric type: {metric_type}. Choose from {list(METRIC_FUNCTIONS.keys())}")
    metric_func = METRIC_FUNCTIONS[metric_type]
    
    results = {metric_type: {}}
    for targ, target_name in zip(targets, target_names):
        results[metric_type][targ] = {}
        # Load feature data
        featuresd = joblib.load(f'{fold_base_path}Fold_0/rf/{target_name}rf2-1_output_std_All.joblib')
        feature_keys = list(featuresd.keys())
        for key in feature_keys:
            results[metric_type][targ][key] = {}
            for model_name, suffix in model_types.items():
                # Concatenate data across folds
                y_true_all = []
                y_pred_all = []
                y_train_all = []
                for fold_num in range(n_folds):
                    pls_dir = f'{fold_base_path}Fold_{fold_num}/rf/'
                    file_path = f'{pls_dir}{target_name}{suffix}.joblib'
                    if not os.path.exists(file_path):
                        print(f"Missing file: {file_path}")
                        continue
                    pls_dict = joblib.load(file_path)
                    if key not in pls_dict:
                        print(f"Feature set {key} not found in {file_path}")
                        continue
                    y_true_all.append(pls_dict[key]['data'][test_t])
                    y_pred_all.append(pls_dict[key]['data'][test_p])
                    if 'yttrain_std' in pls_dict[key]['data']:
                        y_train_all.append(pls_dict[key]['data']['yttrain_std'])
                
                if not y_true_all or not y_pred_all:
                    print(f"No valid data for {targ}, {key}, {model_name}. Skipping.")
                    continue
                
                # Concatenate data
                y_true_all = pd.concat(y_true_all)
                y_pred_all = pd.concat(y_pred_all)
                y_train_all = pd.concat(y_train_all) if y_train_all else None
                
                # Get indices
                indices_all = y_true_all.index
                common_ind = y_true_all.index.intersection(demo.index)
                demo_subset = demo.loc[common_ind]
                aa_indices = demo_subset[demo_subset['race_ethnicity'] == 2].index.tolist()
                wa_indices = demo_subset[demo_subset['race_ethnicity'] == 1].index.tolist()
                all_indices = aa_indices + wa_indices
                
                # Create index mapping for NumPy arrays
                y_true_all = y_true_all.loc[all_indices]
                y_pred_all = y_pred_all.loc[all_indices]
                index_map = {idx: i for i, idx in enumerate(all_indices)}
                aa_pos = [index_map[idx] for idx in aa_indices]
                wa_pos = [index_map[idx] for idx in wa_indices]
                
                # Convert to NumPy arrays
                y_true_array = y_true_all.values.flatten()
                y_pred_array = y_pred_all.values.flatten()
                y_train_array = y_train_all.values.flatten() if y_train_all is not None else None
                
                # Compute metrics on concatenated data
                metric_aa = metric_func(y_true_array[aa_pos], y_pred_array[aa_pos], y_train_array)
                metric_wa = metric_func(y_true_array[wa_pos], y_pred_array[wa_pos], y_train_array)
                diff = metric_aa - metric_wa
                
                # Permutation test
                perm_diffs, perm_p_value = permutation_metric_difference(
                    y_true_array, y_pred_array, aa_indices, wa_indices, metric_func, y_train_array, n_permutations=P
                )
                
                # Compute mean and std for perm_diffs
                perm_diff_mean = np.mean(perm_diffs)
                perm_diff_std = np.std(perm_diffs, ddof=1)
                
                results[metric_type][targ][key][model_name] = {
                    'metric_aa': metric_aa,
                    'metric_wa': metric_wa,
                    'diff': diff,
                    'perm_diff_mean': perm_diff_mean,
                    'perm_diff_std': perm_diff_std,
                    'perm_diff_dist': perm_diffs,
                    'perm_p_value': perm_p_value
                }
    
    # Save results with metric type in filename
    output_dir = os.path.join(fold_base_path, 'rf_results_concat_transformed')
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(results, os.path.join(output_dir, f'stat_results_{metric_type}_xstd.joblib'))
    #print(f"Statistical comparison for {metric} completed and saved.")
    return results

if __name__ == "__main__":
    test_p = 'yptest'
    test_t = 'yttest_std'
    metrics = ['r2', 'cod', 'pearson', 'mse', 'mae']#

    Parallel(n_jobs=-1)(
    delayed(run_statistical_analysis)(metric_type=metric)
    for metric in metrics)
    

## Calculate ethnicity bias index

### Unimodal

In [ ]:
# Paths and settings
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
output_dir = os.path.join(fold_base_path, 'results_concat_transformed', 'tables')
os.makedirs(output_dir, exist_ok=True)

# Metrics and target
metrics = ['r2', 'cod', 'pearson', 'mse', 'mae']
metric_labels = {
    'r2': 'R²',
    'cod': 'pCOD',
    'pearson': "Fisher\'s z",
    'mse': 'MSE',
    'mae': 'MAE'
}
model_types = ['aa', 'wa']  # Only aa and wa models for bias index
target = 'nihtbx_totalcomp_uncorrected'
targ = 'total'
def create_ethnicity_bias_index_table():
    # Collect data across all metrics
    data = {}
    all_keys = set()
    
    # Load results for each metric
    for metric in metrics:
        results_file = os.path.join(fold_base_path, 'results_concat_transformed', f'stat_results_{metric}.joblib')
        if not os.path.exists(results_file):
            print(f"Results file not found: {results_file}")
            continue
        results = joblib.load(results_file)
        if metric not in results or target not in results[metric]:
            print(f"No data for metric {metric} or target {target}")
            continue
        
        # Extract feature sets across all modalities
        for modality in results[metric][target]:
            for key in results[metric][target][modality]:
                all_keys.add(key)
                if key not in data:
                    data[key] = {}
                for model in model_types:
                    if model not in data[key]:
                        data[key][model] = {}
                    if model in results[metric][target][modality][key]:
                        metric_aa = results[metric][target][modality][key][model]['metric_aa']
                        metric_wa = results[metric][target][modality][key][model]['metric_wa']
                        data[key][model][metric] = {
                            'metric_aa': metric_aa,
                            'metric_wa': metric_wa
                        }

    if not data:
        print("Error: No valid data loaded")
        return None
    
    # Prepare table data
    table_data = []
    for key in sorted(all_keys):  # Alphabetical order
        row = {'Feature Set': key}
        for metric in metrics:

            bias_index = np.nan  # Default if data missing
            All_perf_index = np.nan  # Default if data missing
            if key in data and all(model in data[key] and metric in data[key][model] for model in model_types):
                # Compute AA-WA differences for each model
                diff_aa = data[key]['aa'][metric]['metric_aa'] - data[key]['aa'][metric]['metric_wa']
                diff_wa = data[key]['wa'][metric]['metric_aa'] - data[key]['wa'][metric]['metric_wa']
                # Ethnicity Bias Index
                bias_index = diff_aa - diff_wa
            row[metric] = bias_index

        
        # table_data.append(row)
        modality_chunks = ['abccConn','abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']
        All_all_dict0 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abcdCntr_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abccConn_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abccSmri_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abccCntr_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abcdRsmri_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/{targ}_abccGtfc_pls_output_std_All.joblib'),
                            
                        }
        All_all_dict1 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abcdCntr_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abccConn_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abccSmri_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abccCntr_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abcdRsmri_pls_output_std_All.joblib'),
                            **joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/{targ}_abccGtfc_pls_output_std_All.joblib'),
                        }

        All_perf_index1 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['mae'], All_all_dict1[key]['model']['perf'].loc['test']['mae']))
        All_perf_index2 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['r2'], All_all_dict1[key]['model']['perf'].loc['test']['r2']))
        # All_perf_index3 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['cod'], All_all_dict1[key]['model']['perf'].loc['test']['cod']))        
        All_perf_index4 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['nmse'], All_all_dict1[key]['model']['perf'].loc['test']['nmse']))
        row['mae_all'] = All_perf_index1
        row['r2_all'] = All_perf_index2
        # row['cod_all'] = All_perf_index3
        row['nmse_all'] = All_perf_index4
        table_data.append(row)

    if not table_data:
        print("Error: No valid data for table")
        return None
    
    # Create DataFrame
    df = pd.DataFrame(table_data)
    
    # Select columns for table
    columns = ['Feature Set'] + metrics + ['mae_all']
    table_df = df[columns]
    
    # Save as CSV
    csv_file = os.path.join(output_dir, target + '_ethnicity_bias_index_all_features_All.csv')
    table_df.to_csv(csv_file, index=False, float_format='%.3f')
    print(f"Table saved to {csv_file}")
    
    return table_df

def main():
    print("Generating ethnicity bias index table for all feature sets...")
    table_df = create_ethnicity_bias_index_table()
    if table_df is not None:
        print(table_df)
        print("\n")

if __name__ == "__main__":
    main()

Generating ethnicity bias index table for all feature sets...
Table saved to /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/results_concat_transformed/tables/nihtbx_totalcomp_uncorrected_ethnicity_bias_index_all_features_waaa.csv
              Feature Set        r2       cod   pearson       mse       mae  \
0        Avg_T1_ASEG_Vol_  2.435197  2.711523 -0.015909 -2.711523 -0.947885   
1            Avg_T2_ASEG_  2.630397  2.751174  0.040902 -2.751174 -0.968540   
2                     DTI  2.598139  2.578330 -0.005918 -2.578330 -0.944987   
3              Dest_Area_  2.549225  2.654064  0.030631 -2.654064 -0.941109   
4             Dest_Thick_  2.352133  2.519828  0.002493 -2.519828 -0.888335   
..                    ...       ...       ...       ...       ...       ...   
82  rsmri_within_avg_data  1.929046  2.051050  0.051651 -2.051050 -0.744596   
83                   subc  2.113961  2.134916 -0.028223 -2.134916 -0.803699   
84            subnet_rest 

### Multimodal

In [ ]:
# Paths and settings
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
output_dir = os.path.join(fold_base_path, 'rf_results_concat_transformed', 'tables')
os.makedirs(output_dir, exist_ok=True)

# Metrics and target
metrics = ['r2', 'cod', 'pearson', 'mse', 'mae']
metric_labels = {
    'r2': 'R²',
    'cod': 'pCOD',
    'pearson': "Fisher\'s z",
    'mse': 'MSE',
    'mae': 'MAE'
}
model_types = ['aa', 'wa']  # Only aa and wa models for bias index
target = 'nihtbx_totalcomp_uncorrected'
targ = 'total'
def create_ethnicity_bias_index_table():
    # Collect data across all metrics
    data = {}
    all_keys = set()
    
    # Load results for each metric
    for metric in metrics:
        results_file = os.path.join(fold_base_path, 'rf_results_concat_transformed', f'stat_results_{metric}.joblib')
        if not os.path.exists(results_file):
            print(f"Results file not found: {results_file}")
            continue
        results = joblib.load(results_file)
        print(results.keys())
        if metric not in results or target not in results[metric]:
            print(f"No data for metric {metric} or target {target}")
            continue
        
        # Extract feature sets across all modalities

        for key in results[metric][target]:
            print(key)
            all_keys.add(key)
            if key not in data:
                data[key] = {}
            for model in model_types:
                if model not in data[key]:
                    data[key][model] = {}
                if model in results[metric][target][key]:
                    metric_aa = results[metric][target][key][model]['metric_aa']
                    metric_wa = results[metric][target][key][model]['metric_wa']
                    data[key][model][metric] = {
                        'metric_aa': metric_aa,
                        'metric_wa': metric_wa
                    }

    if not data:
        print("Error: No valid data loaded")
        return None
    
    # Prepare table data
    table_data = []
    for key in sorted(all_keys):  # Alphabetical order
        row = {'Feature Set': key}
        print(key)
        for metric in metrics:

            bias_index = np.nan  # Default if data missing
            All_perf_index = np.nan  # Default if data missing
            if key in data and all(model in data[key] and metric in data[key][model] for model in model_types):
                # Compute AA-WA differences for each model
                diff_aa = data[key]['aa'][metric]['metric_aa'] - data[key]['aa'][metric]['metric_wa']
                diff_wa = data[key]['wa'][metric]['metric_aa'] - data[key]['wa'][metric]['metric_wa']
                # Ethnicity Bias Index
                bias_index = diff_aa - diff_wa
            row[metric] = bias_index

        
        modality_chunks = ['abccConn','abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']
        # All_all_dict0 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/rf/{targ}_rf2-1_output_std_All.joblib'),
        #                   }
        All_all_dict0 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/rf/{targ}_rf2-1_output_std_All.joblib'),
                          }
        print(All_all_dict0.keys())
        # All_all_dict1 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/rf/{targ}_rf2-1_output_std_All.joblib'),
        #                }
        All_all_dict1 = {**joblib.load(f'/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/rf/{targ}_rf2-1_output_std_All.joblib'),
                       }
        All_perf_index1 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['mae'], All_all_dict1[key]['model']['perf'].loc['test']['mae']))
        All_perf_index2 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['r2'], All_all_dict1[key]['model']['perf'].loc['test']['r2']))
        # All_perf_index3 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['cod'], All_all_dict1[key]['model']['perf'].loc['test']['cod']))        
        All_perf_index4 = np.mean((All_all_dict0[key]['model']['perf'].loc['test']['nmse'], All_all_dict1[key]['model']['perf'].loc['test']['nmse']))
        row['mae_all'] = All_perf_index1
        row['r2_all'] = All_perf_index2
        # row['cod_all'] = All_perf_index3
        row['nmse_all'] = All_perf_index4
        table_data.append(row)

    if not table_data:
        print("Error: No valid data for table")
        return None
    
    # Create DataFrame
    df = pd.DataFrame(table_data)
    
    # Select columns for table
    columns = ['Feature Set'] + metrics + ['mae_all']
    table_df = df[columns]
    
    # Save as CSV
    csv_file = os.path.join(output_dir, target + '_ethnicity_bias_index_all_features.csv')
    table_df.to_csv(csv_file, index=False, float_format='%.3f')
    print(f"Table saved to {csv_file}")
    
    return table_df

def main():
    print("Generating ethnicity bias index table for all feature sets...")
    table_df = create_ethnicity_bias_index_table()
    if table_df is not None:
        print(table_df)
        print("\n")

if __name__ == "__main__":
    main()

Generating ethnicity bias index table for all feature sets...
dict_keys(['r2'])
nAll1_abcd
nGD_WMCntr
nGD_MidCntr
GD_SstCntr
nGD_TaskCntr
AllFCs
AllRest
AllSmri
NonTask
nGD_TaskAll
nGD_WMAll
nGD_MidAll
nGD_SstAll
D_WMCntr
D_MidCntr
D_SstCntr
D_TaskCntr
D_TaskAll
nG_WmAll
nG_MidAll
nG_SstAll
nG_TaskCntr
nD_TaskCntr
nD_WmAll
nD_MidAll
nD_SstAll
PrConn
TaskConn
dict_keys(['cod'])
nAll1_abcd
nGD_WMCntr
nGD_MidCntr
GD_SstCntr
nGD_TaskCntr
AllFCs
AllRest
AllSmri
NonTask
nGD_TaskAll
nGD_WMAll
nGD_MidAll
nGD_SstAll
D_WMCntr
D_MidCntr
D_SstCntr
D_TaskCntr
D_TaskAll
nG_WmAll
nG_MidAll
nG_SstAll
nG_TaskCntr
nD_TaskCntr
nD_WmAll
nD_MidAll
nD_SstAll
PrConn
TaskConn
dict_keys(['pearson'])
nAll1_abcd
nGD_WMCntr
nGD_MidCntr
GD_SstCntr
nGD_TaskCntr
AllFCs
AllRest
AllSmri
NonTask
nGD_TaskAll
nGD_WMAll
nGD_MidAll
nGD_SstAll
D_WMCntr
D_MidCntr
D_SstCntr
D_TaskCntr
D_TaskAll
nG_WmAll
nG_MidAll
nG_SstAll
nG_TaskCntr
nD_TaskCntr
nD_WmAll
nD_MidAll
nD_SstAll
PrConn
TaskConn
dict_keys(['mse'])
nAll1_abcd
nGD_W

## Plot: Performance bar plots

### Unimodal

In [ ]:
### Unimodal bar plots - test AA and WA mae (or r2, pearson r) across different training strategies
import matplotlib

# Paths and settings
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
targets = ['nihtbx_totalcomp_uncorrected']
modality_chunks = ['abccConn', 'abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']

# model display order
model_types = {'All': '_pls_output_std_All', 'wa': '_pls_output_std_wa',
               'aa': '_pls_output_std_aa', 'aawa': '_pls_output_std_waaa',
               }
model_labels = {
    "All": "All",
    "wa": "RandWA-only",
    "aa": "AA-only",
    "aawa": "Balanced AA+RandWA",
}

model_order = list(model_types.keys())
metric_type = 'mae'#'pearson'#'r2'
metric_label = 'MAE'#'R2'#'Pearson' #'Pearson' R''MAE'

# main function
def plot_mae_difference_barplots(results_file, target, models_to_plot=None):
    if models_to_plot is None:
        models_to_plot = model_order

    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    results = joblib.load(results_file)
    if metric_type not in results or target not in results[metric_type]:
        print(f"No data for metric {metric_type} or target {target}")
        return

    # Collect rows
    rows = []
    for modality in modality_chunks:
        if modality not in results[metric_type][target]:
            continue
        for feat, feat_data in results[metric_type][target][modality].items():
            if feat not in l1_labels:
                continue
            for model in model_order:
                if model not in feat_data:
                    continue
                mae_aa = feat_data[model]['metric_aa']
                mae_wa = feat_data[model]['metric_wa']
                diff = mae_aa - mae_wa if model in models_to_plot else 0
                p_val = feat_data[model]['perm_p_value'] if model in models_to_plot else float('nan')
                # categorise
                if modality in ['abccConn', 'abccGtfc']:
                    group = 'Conn+Gtfc'
                elif modality == 'abcdRsmri':
                    group = 'Conn+Gtfc' if 'rsmri' in feat.lower() else 'Smri'
                elif modality == 'abccSmri':
                    group = 'Smri'
                elif modality == 'abccCntr':
                    group = 'abccCntr'
                elif modality == 'abcdCntr':
                    group = 'abcdCntr'
                else:
                    group = modality

                rows.append({
                    'Feature': feat,
                    'Feature Label': l1_labels[feat],
                    'Model': model,
                    'Group': group,
                    'Diff': diff,
                    #''
                    'p_val': p_val,
                    'aa_mae': mae_aa,
                    'wa_mae': mae_wa
                })

    df = pd.DataFrame(rows).drop_duplicates(subset=['Feature', 'Model', 'Group'], keep='first')

    # Y-axis limits
    # min_diff, max_diff = -0.20, 0.40 # pear r
    min_diff, max_diff = 0, 1.5 #-1.50, 1.0 # r2
    margin = (max_diff - min_diff) * 0.05
    ylim = (min_diff - margin, max_diff + margin)

    groups = {
        'Conn+Gtfc': "FCs",
        'Smri': "sMRI",
        'abccCntr': "ABCC Contrasts",
        'abcdCntr': "ABCD Contrasts"
    }

    base_palette = sns.color_palette("Set2", n_colors=len(model_order))
    palette = {m: c if m in models_to_plot else 'white' for m, c in zip(model_order, base_palette)}

    output_dir = os.path.join(fold_base_path, 'plots_ppt')
    os.makedirs(output_dir, exist_ok=True)
    model_suffix = '_'.join(models_to_plot).replace(' ', '_')

    for grp_key, title in groups.items():
        sub = df[df['Group'] == grp_key].copy()
        if sub.empty:
            continue

        # Use features from l1_labels, filtered by group and present in sub
        features_order = [feat for feat in l1_labels.keys() if feat in sub['Feature'].unique()]
        if not features_order:
            print(f"No valid features for group {grp_key}, skipping")
            continue
        feature_labels = [l1_labels[feat] for feat in features_order]

        # Subplot layout: stacked, no space between
        fig, axes = plt.subplots(
            2, 1,
            figsize=(0.75 * len(features_order), 9),
            sharex=True,
            gridspec_kw={'hspace': 0}  #remove vertical space
        )

        metrics = [('aa_mae', 'AA'), ('wa_mae', 'WA')]

        # lookup tables
        p_lookup = sub.set_index(['Feature', 'Model'])['p_val']
        diff_lookup = sub.set_index(['Feature', 'Model'])['Diff']
        feature_map = {l1_labels[feat]: feat for feat in features_order}

        for ax, (metric_col, label) in zip(axes, metrics):
            sns.set_theme(style="white")
            bar_kwargs = dict(
                data=sub,
                x="Feature",
                y=metric_col,
                hue="Model",
                hue_order=model_order,
                order=features_order,
                palette=palette,
                width=0.8,
                errorbar=None
            )
            try:
                sns.barplot(**bar_kwargs, ax=ax)
            except TypeError:
                bar_kwargs.pop('errorbar', None)
                sns.barplot(ci=None, ax=ax, **bar_kwargs)

            ax.set_xticks(range(len(features_order)))
            ax.set_xticklabels(feature_labels, fontsize=20, rotation=60, ha='right')
            ax.tick_params(axis='y', labelsize=20)
            ax.set_xlim(-0.5, len(features_order) - 0.5)
            for i in range(len(features_order) - 1):
                ax.axvline(x=i + 0.5, color='grey', linestyle='--', linewidth=0.5, alpha=0.99)
            ax.set_ylim(ylim)
            yticks = np.arange(0.5, 1.6, 0.5)
            ax.set_yticks(yticks)
            ax.set_yticklabels([f"{t:.1f}" for t in yticks], fontsize=20)

            # small tick marks (both major and minor)
            ax.tick_params(axis='y', which='major', length=6, width=1, direction='inout', labelsize=20)
            ax.tick_params(axis='y', which='minor', length=3, width=0.8, direction='inout')

            # Add minor ticks 
            ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator(2))
            ax.set_ylabel(f"", fontsize=12)
            ax.axhline(0, lw=1, ls="-", color="Grey", alpha=0.4)
            # ax.set_yticks(np.arange(0.5, 1.6, 0.5))
            # ax.invert_yaxis()
            # remove the title for the top plot
            # if label == "AA":
            #     pass  # do nothing
            # else:
            ax.get_legend().remove()
            if label == "AA":
                ax.spines['bottom'].set_color("white")
            elif label == "WA":
                ax.spines['top'].set_color("white")
            # #legend
            # if label == "AA":
            #     handles, labels = ax.get_legend_handles_labels()
            #     new_labels = [model_labels.get(l, l) for l in labels]  # map to new names
            #     ax.legend(
            #         handles, new_labels,
            #         title="Training Set",
            #         bbox_to_anchor=(1.02, 1),
            #         loc="upper left",
            #         frameon=True
            #     )
        


            # --- star annotations (for both AA and WA) ---
            bar_containers = [c for c in ax.containers if isinstance(c, matplotlib.container.BarContainer)]
            bar_containers = bar_containers[:len(model_order)]
            y_min, y_max = ax.get_ylim()
            y_range = max_diff-min_diff#y_max - y_min
            offset = 0.01 * y_range  # bigger offset for visibility

            for h_idx, container in enumerate(bar_containers):
                model = model_order[h_idx]
                for j, bar in enumerate(container.patches):
                    if j >= len(features_order):
                        continue
                    feat = features_order[j]  # directly use feature ID instead of xticklabels
                    if (feat, model) not in p_lookup:
                        continue
                    pval = p_lookup[(feat, model)]
                    diff = diff_lookup[(feat, model)]
                    if pd.notna(pval) and pval < 0.05:
                        x = bar.get_x() + bar.get_width() / 2
                        print('pvalis')
                        y = bar.get_height()
                        star_color = 'blue' if diff < 0 else 'red'#'black'
                    if label == "WA":
                        axes[1].text(x, -0.02, "★", va="bottom", ha="center", fontsize=12, color=star_color, zorder=10) # mae
                        # axes[1].text(x, 1.01, "★", va="bottom", ha="center", fontsize=12, color=star_color, zorder=10) # r2
                        #axes[1].text(x, 0.40, "★", va="bottom", ha="center", fontsize=12, color=star_color, zorder=10) # pearsons r

            # Flip the WA plot vertically
            if label == "WA":
                ax.invert_yaxis()
                yticks = np.arange(0.5, 1.6, 0.5)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{t:.1f}" for t in yticks], fontsize=20)
                ax.tick_params(axis='y', which='major', length=6, width=1, direction='inout', labelsize=20)
                ax.tick_params(axis='y', which='minor', length=3, width=0.8, direction='inout')
                ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator(2))


        axes[1].set_xlabel("", fontsize=12)


        plt.tight_layout()
        outpath = os.path.join(output_dir, f'{target}_{metric_type}_AA_WA_barplot_{grp_key}_{model_suffix}.png')
        plt.savefig(outpath, dpi=600, bbox_inches='tight')
        outpath_svg = os.path.join(output_dir, f'{target}_{metric_type}_AA_WA_barplot_{grp_key}_{model_suffix}.svg')
        plt.savefig(outpath_svg, bbox_inches='tight')
        plt.close()
        print(f"Saved plot: {outpath}")



if __name__ == "__main__":
    results_file = os.path.join(fold_base_path, 'results_concat_transformed', f'stat_results_{metric_type}.joblib')
    for target in targets:
        plot_mae_difference_barplots(results_file, target, models_to_plot=['All', 'wa', 'aa', 'aawa'])


### Multimodal

In [ ]:
### Multimodal bar plots - test AA and WA mae (or r2, pearson r) across different training strategies
## plot stacked paper , aa and wa maes - inversed

# Paths and settings
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
targets = ['nihtbx_totalcomp_uncorrected']
Target_name = 'Total Cognitive Functioning Score'
modality_chunks = ['abccConn', 'abccSmri', 'abccCntr', 'abcdRsmri', 'abcdCntr', 'abccGtfc']

# desired model display order
model_types = {'All': '_pls_output_std_All', 'wa': '_pls_output_std_wa',
               'aa': '_pls_output_std_aa', 'aawa': '_pls_output_std_waaa'}
model_order = list(model_types.keys())

metric_type = 'mae'#'pearson'#'r2'
metric_label = 'MAE'#'Pearson'#'R2'



def plot_mae_difference_barplots(results_file, target, models_to_plot=None):
    if models_to_plot is None:
        models_to_plot = model_order

    if not os.path.exists(results_file):
        print(f"Results file not found: {results_file}")
        return

    results = joblib.load(results_file)
    if metric_type not in results or target not in results[metric_type]:
        print(f"No data for metric {metric_type} or target {target}")
        return

    rows_plot = []
    for feat, feat_data in results[metric_type][target].items():
        if feat not in stacked_labels:
            continue
        for model in model_order:
            if model not in feat_data:
                continue
            mae_aa = feat_data[model]['metric_aa']
            mae_wa = feat_data[model]['metric_wa']
            diff = mae_aa - mae_wa
            p_val = feat_data[model]['perm_p_value']
            if model in models_to_plot:
                rows_plot.append({
                    'Feature': feat,
                    'Feature Label': stacked_labels[feat],
                    'Model': model,
                    'Diff': diff,
                    'p_val': p_val,
                    'aa_mae': mae_aa,
                    'wa_mae': mae_wa
                })
    df = pd.DataFrame(rows_plot).drop_duplicates(subset=['Feature', 'Model'], keep='first')

    features_order = [feat for feat in stacked_labels.keys() if feat in df['Feature'].unique()]
    feature_labels = [stacked_labels[feat] for feat in features_order]

    
    min_diff, max_diff = 0, 1.5
    margin = (max_diff - min_diff) * 0.05
    ylim = (min_diff - margin, max_diff + margin)
    yticks = np.arange(0.5, 1.5, 0.5)

    # pastel palette
    base_palette = sns.color_palette("Set2", n_colors=len(model_order))
    palette = {m: c if m in models_to_plot else 'white' for m, c in zip(model_order, base_palette)}

    output_dir = os.path.join(fold_base_path, 'plots')
    os.makedirs(output_dir, exist_ok=True)

    model_suffix = '_'.join(models_to_plot).replace(' ', '_')

    sub = df.copy()
    if sub.empty:
        return

    fig, axes = plt.subplots(
        2, 1,
        figsize=(0.75 * len(features_order), 9),
        sharex=True,
        gridspec_kw={'hspace': 0}
    )

    metrics = [('aa_mae', 'AA'), ('wa_mae', 'WA')]
    p_lookup = sub.set_index(['Feature', 'Model'])['p_val']
    diff_lookup = sub.set_index(['Feature', 'Model'])['Diff']

    for ax, (metric_col, label) in zip(axes, metrics):
        sns.set_theme(style="white")
        bar_kwargs = dict(
            data=sub,
            x="Feature",
            y=metric_col,
            hue="Model",
            hue_order=model_order,
            order=features_order,
            palette=palette,
            width=0.8,
            errorbar=None
        )
        try:
            sns.barplot(**bar_kwargs, ax=ax)
        except TypeError:
            bar_kwargs.pop('errorbar', None)
            sns.barplot(ci=None, ax=ax, **bar_kwargs)

        # axes, ticks, separators 
        ax.set_xticks(range(len(features_order)))
        ax.set_xticklabels(feature_labels, fontsize=20, rotation=60, ha='right')
        ax.tick_params(axis='y', which='major', length=6, width=1, direction='inout', labelsize=20)
        ax.tick_params(axis='y', which='minor', length=3, width=0.8, direction='inout')
        ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator(2))
        ax.set_xlim(-0.5, len(features_order) - 0.5)
        for i in range(len(features_order) - 1):
            ax.axvline(x=i + 0.5, color='grey', linestyle='--', linewidth=0.5, alpha=0.9)
        ax.set_ylim(ylim)
        ax.set_yticks(yticks)
        ax.set_yticklabels([f"{t:.1f}" for t in yticks], fontsize=20)
        ax.axhline(0, lw=1, ls="-", color="Grey", alpha=0.4)
        ax.get_legend().remove()

        # invert WA plot
        if label == "WA":
            ax.invert_yaxis()

        # remove spines between plots
        if label == "AA":
            ax.spines['bottom'].set_color("white")
        elif label == "WA":
            ax.spines['top'].set_color("white")

        #  significance stars 
        bar_containers = [c for c in ax.containers if isinstance(c, matplotlib.container.BarContainer)]
        bar_containers = bar_containers[:len(model_order)]
        y_range = max_diff - min_diff
        offset = 0.01 * y_range

        for h_idx, container in enumerate(bar_containers):
            model = model_order[h_idx]
            for j, bar in enumerate(container.patches):
                if j >= len(features_order):
                    continue
                feat = features_order[j]
                if (feat, model) not in p_lookup:
                    continue
                pval = p_lookup[(feat, model)]
                diff = diff_lookup[(feat, model)]
                if pd.notna(pval) and pval < 0.05:
                    x = bar.get_x() + bar.get_width() / 2
                    y = bar.get_height()
                    star_color = 'blue' if diff < 0 else 'red'#'black'
                    if label == "WA":
                        axes[1].text(x, -0.02, "★", va="bottom", ha="center", fontsize=12, color=star_color, zorder=10)

    axes[1].set_xlabel("Stacked Sets of Feature", fontsize=14)
    plt.tight_layout()

    outpath = os.path.join(output_dir, f'{Target_name}_{metric_type}_AA_WA_barplot_stacked_{model_suffix}.png')
    plt.savefig(outpath, dpi=600, bbox_inches='tight')
    
    outpath_svg = os.path.join(output_dir, f'{Target_name}_{metric_type}_AA_WA_barplot_stacked_{model_suffix}.svg')
    plt.savefig(outpath_svg, bbox_inches='tight')

    plt.close()
    print(f"Saved plot: {outpath}")


if __name__ == "__main__":
    results_file = os.path.join(fold_base_path, 'rf_results_concat_transformed', f'stat_results_{metric_type}.joblib')
    for target in targets:

        plot_mae_difference_barplots(results_file, target, models_to_plot=['All', 'wa', 'aa', 'aawa'])


## Plot: bias vs performance

In [ ]:
# scatter plot
# Paths
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
single_dir = os.path.join(fold_base_path, 'results_concat_transformed', 'tables')
stacked_dir = os.path.join(fold_base_path, 'rf_results_concat_transformed', 'tables')
single_csv = os.path.join(single_dir, 'nihtbx_totalcomp_uncorrected_ethnicity_bias_index_all_features.csv')
stacked_csv = os.path.join(stacked_dir, 'nihtbx_totalcomp_uncorrected_ethnicity_bias_index_all_features.csv')
output_plot = os.path.join(single_dir, 'scatter_mae_all_vs_abs_mae_combined_navy.png')
output_plot_svg = os.path.join(single_dir, 'scatter_mae_all_vs_abs_mae_combined_navy.svg')


# Group assignment for coloring (unimodal models only)
def assign_group(feature):
    if feature.startswith(("conn_", "gfc", "tfc", "rsmri", 'rest')):
        return "FCs"
    elif feature.startswith(("cntr_", "artr_")):
        return "Task Contrasts (Glasser)"
    elif feature.startswith(("anti", "feed", "incorrect", "correct", "emotion", "posface", "negface", "place", "X")):
        return "Task Contrasts (Destrieux)"
    else:
        return "sMRI"

# Group colors for unimodal models
group_colors = {
    "FCs": "tab:blue",
    "Task Contrasts (Glasser)": "tab:orange",
    "Task Contrasts (Destrieux)": "tab:green",
    "sMRI": "tab:red"
}

def create_scatter_plot():
    # Load CSVs
    df_single = pd.read_csv(single_csv)
    df_stacked = pd.read_csv(stacked_csv)
    
    # Filter features based on dictionary keys
    df_single = df_single[df_single['Feature Set'].isin(l1_labels.keys())]
    df_stacked = df_stacked[df_stacked['Feature Set'].isin(stacked_labels.keys())]
    
    # Map feature sets to labels
    df_single['Feature Label'] = df_single['Feature Set'].map(l1_labels)
    df_stacked['Feature Label'] = df_stacked['Feature Set'].map(stacked_labels)
    
    df_single['Model Type'] = 'Single'
    df_stacked['Model Type'] = 'Stacked'

    # Combine
    df = pd.concat([df_single, df_stacked], ignore_index=True)
    df['mae_abs'] = df['mae'].abs()

    # Assign groups for single models
    df.loc[df['Model Type'] == 'Single', 'Group'] = df.loc[df['Model Type'] == 'Single', 'Feature Set'].astype(str).apply(assign_group)

    # Pearson correlation
    valid_data = df[['mae_all', 'mae_abs']].dropna()
    corr_text = f'Pearson r = {pearsonr(valid_data["mae_all"], valid_data["mae_abs"])[0]:.3f}' if len(valid_data) >= 2 else 'Pearson r = N/A'

    # Plot
    plt.figure(figsize=(12, 12))
    
    # Single models: colored by group
    for group, subdf in df[df['Model Type'] == 'Single'].groupby('Group'):
        plt.scatter(
            subdf['mae_all'], subdf['mae_abs'],
            color=group_colors.get(group, 'gray'),
            marker='o',
            edgecolors=group_colors.get(group, 'gray'),
            s=300,
            alpha=0.7,
            label=f"Single - {group}"
        )
    
    # Stacked models: navy triangles
    stacked_data = df[df['Model Type'] == 'Stacked']
    plt.scatter(
        stacked_data['mae_all'], stacked_data['mae_abs'],
        color='navy',
        marker='^',
        edgecolors='navy',
        s=300,
        alpha=0.7,
        label='Stacked Models'
    )
    
    # # Labels for stacked models only
    # for i, row in stacked_data.iterrows():
    #     if pd.notna(row['mae_all']) and pd.notna(row['mae_abs']):
    #         label = row['Feature Label']  # Use mapped label
    #         plt.text(row['mae_all'] + 0.005, row['mae_abs'] + 0.012, label, fontsize=8, ha='right', va='top', alpha=0.7)
    
    # Correlation text
    plt.text(0.05, 0.95, corr_text, transform=plt.gca().transAxes, fontsize=22, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.xlabel('MAE (All Model Performance)', fontsize=12)
    plt.ylabel('Absolute Ethnicity Bias Index (MAE)', fontsize=12)
    plt.title('MAE All vs. Absolute Ethnicity Bias Index (Single + Stacked Models)', fontsize=14)
    plt.tick_params(axis='both', labelsize=26)
    x_ticks = np.linspace(0.65, 0.80, 4)
    plt.xticks(x_ticks)    
    y_ticks = [0.4, 0.6, 0.8, 1, 1.2]#np.linspace(0.40, 1.20, 4)
    plt.yticks(y_ticks)   
    # plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20, title="Model Type", title_fontsize=20)
    # plt.grid(True, linestyle='--', alpha=0.7)
    
    # Save
    plt.savefig(output_plot, dpi=600, bbox_inches='tight')
    plt.savefig(output_plot_svg, bbox_inches='tight')
    print(f"Saved plot: {output_plot}")
    
    plt.show()
    plt.close()
    return True

if __name__ == "__main__":
    print("Generating scatter plot ")
    create_scatter_plot()

## Plot: bias ranking

In [ ]:
# rank bar plot 
from matplotlib import font_manager


# Paths
root_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
fold_base_path = root_dir + 'tsp/'
single_dir = os.path.join(fold_base_path, 'results_concat_transformed', 'tables')
stacked_dir = os.path.join(fold_base_path, 'rf_results_concat_transformed', 'tables')
single_csv = os.path.join(single_dir, 'nihtbx_totalcomp_uncorrected_ethnicity_bias_index_all_features_waaa.csv')
stacked_csv = os.path.join(stacked_dir, 'nihtbx_totalcomp_uncorrected_ethnicity_bias_index_all_features_waaa.csv')
output_plot = os.path.join(single_dir, 'barh_bias_ranking_single_stacked_colored-300.png')
output_plot_svg = os.path.join(single_dir, 'barh_bias_ranking_single_stacked_colored.svg')


# Group assignment for unimodal features
def assign_group(feature):
    if feature.startswith(("conn_", "gfc", "tfc", "rsmri", 'rest')):
        return "FCs"
    elif feature.startswith(("cntr_", "artr_")):
        return "Task Contrasts (ABCC)"
    elif feature.startswith(("anti", "feed", "incorrect", "correct", "emotion", "posface", "negface", "place", "X")):
        return "Task Contrasts (ABCD)"
    else:
        return "sMRI"

# Group colors for unimodal features (same as scatter plot)
group_colors = {
    "FCs": "tab:blue",
    "Task Contrasts (ABCC)": "tab:orange",
    "Task Contrasts (ABCD)": "tab:green",
    "sMRI": "tab:red"
}

def create_bias_ranking_plots():
    # Load CSVs
    df_single = pd.read_csv(single_csv)
    df_stacked = pd.read_csv(stacked_csv)
    
    # Filter based on dictionary keys
    df_single = df_single[df_single['Feature Set'].isin(l1_labels.keys())]
    df_stacked = df_stacked[df_stacked['Feature Set'].isin(stacked_labels.keys())]
    
    # Map to labels
    df_single['Feature Label'] = df_single['Feature Set'].map(l1_labels)
    df_stacked['Feature Label'] = df_stacked['Feature Set'].map(stacked_labels)
    
    # Assign groups for unimodal features
    df_single['Group'] = df_single['Feature Set'].apply(assign_group)
    
    # Compute abs for sorting
    df_single['abs_mae'] = df_single['mae'].abs()
    df_stacked['abs_mae'] = df_stacked['mae'].abs()
    
    # Sort by abs_mae descending
    df_single = df_single.sort_values('abs_mae', ascending=False)
    df_stacked = df_stacked.sort_values('abs_mae', ascending=False)
    df_all = pd.concat([df_single, df_stacked], ignore_index=True)
    df_all.to_csv(single_dir + '/ranking_single_stacked.csv')
    # Find global min/max for mae to set common xlim
    all_mae = pd.concat([df_single['mae'].abs(), df_stacked['mae'].abs()])
    min_mae = all_mae.min()
    max_mae = all_mae.max()
    margin = (max_mae - min_mae) * 0.05  # 5% margin
    xlim = (min_mae - margin, max_mae + margin)
    
    # Create figure with two subplots side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(len(df_single), len(df_stacked)) * 0.30), sharey=False)
    
    # Plot for unimodal features (colored by group)
    y_single = np.arange(len(df_single))
    colors_single = [group_colors[group] for group in df_single['Group']]
    ax1.barh(y_single, df_single['mae'].abs(), color=colors_single, alpha=0.7)
    ax1.set_yticks(y_single)
    ax1.set_yticklabels(df_single['Feature Label'],  fontsize=16)#fontfamily='DejaVu Sans',
    ax1.tick_params(axis='x', labelsize=20, labelrotation=0)
    ax1.set_xlim(xlim)
    ax1.set_ylim(-1.1, len(df_single) - 0.0)  # unimodal features
    # ax1.set_xlabel('Absolute Ethnicity Bias Index (MAE)', fontsize=14)
    ax1.set_title('')
    ax1.axvline(0, color='black', linestyle='--', linewidth=0.5)  # Zero line
    ax1.invert_yaxis()  # Highest rank at top
    ax1.axvline(0.5, color='grey', lw=0.8, linestyle='--', alpha=0.6)
    ax1.axvline(1, color='grey', lw=0.8, linestyle='--', alpha=0.6)

    # Add legend for unimodal features
    # legend_elements = [plt.Line2D([0], [0], color=color, lw=4, label=group) for group, color in group_colors.items()]
    # ax1.legend(handles=legend_elements, title="Feature Group", loc='upper right', fontsize=10, title_fontsize=11)
    
    # Plot for stacked features (navy)
    y_stacked = np.arange(len(df_stacked))
    ax2.barh(y_stacked, df_stacked['mae'].abs(), color='navy')
    ax2.set_yticks(y_stacked)
    ax2.set_yticklabels(df_stacked['Feature Label'], fontsize=16)#fontfamily='DejaVu Sans', 
    ax2.tick_params(axis='x', labelsize=20, labelrotation=0)
    ax2.set_xlim(xlim)
    ax2.set_ylim(-1.0, len(df_stacked) - 0.2)  # Stacked features
    # ax2.set_xlabel('Absolute Ethnicity Bias Index (MAE)', fontsize=14)
    ax2.set_title('')
    ax2.axvline(0, color='black', linestyle='--', linewidth=0.5)  # Zero line
    ax2.invert_yaxis()  # Highest rank at top
    ax2.axvline(0.5, color='grey', lw=0.8, linestyle='--', alpha=0.6)
    ax2.axvline(1, color='grey', lw=0.8, linestyle='--', alpha=0.6)


    # Adjust layout
    plt.tight_layout()
    
    # Save
    plt.savefig(output_plot, dpi=300)
    plt.savefig(output_plot_svg, bbox_inches='tight')
    print(f"Saved plot: {output_plot}")
    
    plt.show()
    plt.close()
    return True

if __name__ == "__main__":
    print("Generating bias ranking bar plots...")
    create_bias_ranking_plots()

## Plot: incremental sampling/oversampling

In [ ]:
# line plots showing mae changes as AAs are added - 10 most and least biased
# Paths and settings
fold_num = 0  # or 1
pls_dirs = {
    'Single': '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/sim/simulation/',
    'Stacked': '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/rf/simulation/'
}

features_names_single = ['abcdRsmri', 'abcdCntr', 'abccSmri', 'abccCntr',
                         'conn_mid', 'conn_sst', 'conn_wm', 'conn_rest', 'gfc', 'tfc']#'avg_rest'
features_names_stacked = ['AllFCs', 'AllRest', 'AllSmri', 'GD_SstCntr', 'nAll1_abcd', 'nGD_MidAll',
                          'nGD_MidCntr', 'nGD_SstAll', 'nGD_TaskAll', 'nGD_TaskCntr', 'nGD_WMAll',
                          'nGD_WMCntr', 'NonTask', 'TaskConn']

targ_list = ['total_']

plot_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/trend_plots'
table_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/tables'
rank_table_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/results_concat_transformed/tables/ranking_single_stacked.csv'
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)


# Collect all data
rank_table =  pd.read_csv(rank_table_dir)
# Select bottom 10 and top 10
least_biased = rank_table.nsmallest(10, 'abs_mae')['Feature Set'].tolist()
most_biased = rank_table.nlargest(10, 'abs_mae')['Feature Set'].tolist()
selected_features = least_biased + most_biased
print(selected_features)
all_data = []


for model_type, pls_dir in pls_dirs.items():
    if model_type == 'Single':
        features_names = features_names_single
        print(features_names_single)
    else:
        features_names = features_names_stacked

    for targ in targ_list:
        # Determine file prefix
        if model_type == 'Single':
            target_name = targ
        else:
            target_name = targ

        table_rows = []
        
        for features_name in features_names:

            if model_type == 'Single':
                pls_dict_file = os.path.join(pls_dir, f"{target_name}{features_name}_pls_output_std_Ewaaa_sim.joblib")
            else:
                pls_dict_file = os.path.join(pls_dir, f"{target_name}{features_name}_rf2_output_std_sim_Ewaaa.joblib")

            if not os.path.exists(pls_dict_file):
                print(f"File not found: {pls_dict_file}")
                continue

            pls_dict = joblib.load(pls_dict_file)

            for key in pls_dict:
                if key in selected_features:
                    print(features_name, key)
                    best_aa_round = pls_dict[key].pop('best_aa_round')
                    best_wa_round = pls_dict[key].pop('best_wa_round')

                    iterations = sorted([int(sim_round) for sim_round in pls_dict[key].keys()])
                    test_aa_mae = [pls_dict[key][str(i)]['model']['perf'].loc['test_aa', 'mae'] for i in iterations]
                    test_wa_mae = [pls_dict[key][str(i)]['model']['perf'].loc['test_wa', 'mae'] for i in iterations]

                    # Best MAEs
                    best_aa_idx = np.argmin(test_aa_mae)
                    best_aa_mae = test_aa_mae[best_aa_idx]
                    best_aa_iter = iterations[best_aa_idx]
                    best_aa_per = 100 * (best_aa_iter / (best_wa_round['train_lens']['len_aa'] + best_aa_iter))

                    best_wa_idx = np.argmin(test_wa_mae)
                    best_wa_mae = test_wa_mae[best_wa_idx]
                    best_wa_iter = iterations[best_wa_idx]
                    best_wa_per = 100 * (best_wa_iter / (best_wa_round['train_lens']['len_aa'] + best_wa_iter))

                    percs = [100 * (i / (best_wa_round['train_lens']['len_aa'] + i)) for i in iterations]
                    org_aa_samp = 50

                    # Table row
                    table_rows.append({
                        'Feature': key,
                        'Model Type': model_type,
                        'best_aa_sim': best_aa_per,
                        'best_aa_mae': best_aa_mae,
                        'best_wa_sim': best_wa_per,
                        'best_wa_mae': best_wa_mae
                    })

                    # Data for grid
                    all_data.append({
                        'feature': key,
                        'percs': percs,
                        'test_aa_mae': test_aa_mae,
                        'test_wa_mae': test_wa_mae,
                        'best_aa_per': best_aa_per,
                        'best_wa_per': best_wa_per,
                        'model_type': model_type
                    })
        # Sort all_data by order in selected_features
        feature_order = {feat: i for i, feat in enumerate(selected_features)}
        all_data_sorted = sorted(all_data, key=lambda d: feature_order[d['feature']])

        # Save table CSV per target & model type
        if table_rows:
            df_table = pd.DataFrame(table_rows).set_index('Feature')
            table_file = os.path.join(table_dir, f'aa_percentage_table_{targ}_{model_type}_fold_{fold_num}_Ewaaa.csv')
            df_table.to_csv(table_file)
            print(f"Saved table to {table_file}")
        else:
            print(f"No table data collected for {targ} ({model_type})")
        

# Plot grid figure with Single and Stacked

# Grid dimensions
n_rows, n_cols = 4, 5
figsize = (n_cols * 2.5, n_rows * 2)
fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, sharex=True, sharey=True)
axes = axes.flatten()

# For legend handles
legend_handles = []

for i, data in enumerate(all_data_sorted):
    if i >= n_rows * n_cols:
        break
    ax = axes[i]

    if data['model_type'] == 'Single':
        lw = 1.2
        color_aa, color_wa = 'blue', 'red'
        linestyle = '-'
        label_dict = l1_labels
    else:
        lw = 1.0
        color_aa, color_wa = 'cyan', 'orange'
        linestyle = '-'
        label_dict = stacked_labels

    line_aa, = ax.plot(data['percs'], data['test_aa_mae'], color=color_aa, lw=lw, linestyle=linestyle)
    line_wa, = ax.plot(data['percs'], data['test_wa_mae'], color=color_wa, lw=lw, linestyle=linestyle)
    ax.axvline(50, color='green', lw=0.8, linestyle='-', alpha=1)

    feature_name = data['feature']
    title = label_dict.get(feature_name, feature_name)
    ax.set_title(title, fontsize=10)
    ax.grid(True, lw=0.3)
    ax.tick_params(axis='both', labelsize=12)

    if i == 0:
        legend_handles = [
            line_aa, line_wa,
            plt.Line2D([0], [0], color='cyan', lw=1.0, linestyle='-'),
            plt.Line2D([0], [0], color='orange', lw=1.0, linestyle='-')
        ]


# Remove unused axes
for j in range(i+1, n_rows * n_cols):
    fig.delaxes(axes[j])

# Global labels
fig.text(0.5, 0.04, 'Percentage of AA in Training Sample (%)', ha='center', fontsize=12)
fig.text(0.04, 0.5, 'Mean Absolute Error (MAE)', va='center', rotation='vertical', fontsize=12)
plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
plt.suptitle('', fontsize=16)

# Single legend for all
fig.legend(legend_handles, ['Single AA', 'Single WA', 'Stacked AA', 'Stacked WA'],
           loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=4, fontsize=10)

# Save figure
grid_file = os.path.join(plot_dir, f'mae_trend_grid_all_models_fold_{fold_num}_Ewaaa.png')
plt.savefig(grid_file, bbox_inches='tight', dpi=600)
grid_file_svg = os.path.join(plot_dir, f'mae_trend_grid_all_models_fold_{fold_num}_Ewaaa.png')
plt.savefig(grid_file_svg, bbox_inches='tight')
plt.show()
print(f"Saved full grid figure to {grid_file}")
